# Amazon Search / LTR Project — Week 3 Prompt V2.2
## Precision-First Atomic Attribute Extraction

Prompt V2.1 improved structure, but attribute extraction still over-predicted and often merged several values into one field.

This version changes **only the attribute-extraction/rewrite behavior**.  
The intent taxonomy stays essentially unchanged because V2/V2.1 already improved intent substantially.

### Main V2.2 changes
1. **Precision-first:** if unsure, omit the attribute.
2. **Explicit attributes only:** inferred needs go to `implicit_requirements`, not `attributes`.
3. **Atomic values:** one attribute object = one concept/value.
4. **No duplication:** brand/category/core product/use case never reappear as attributes.
5. **No product-name leakage:** product names cannot be reused as `model`/`product_line` unless the query explicitly provides a real model/code/line.
6. **Rewrite preservation:** keep all exact codes, decimals, sizes, quantities, and explicit product identifiers.

### Development methodology
The same 50 human-labeled queries are the **development set**.  
Only after this version passes the gate should you run the remaining fixed test queries.

### Required uploads
- `amazon_week1_v4_processed_data.zip`
- `amazon_week2_baseline_ranking_results.zip`
- `week3_manual_validation_50_normalized.csv`

In [ ]:
!pip -q install pandas numpy pyarrow rank-bm25 scikit-learn tqdm requests matplotlib

In [ ]:
MODEL = "llama3:8b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
RANDOM_SEED = 42

TARGET_ATTRIBUTE_F1 = 0.75
TARGET_INTENT_ACCURACY = 0.60
MAX_RECALL50_RELATIVE_DROP = 0.02

RUN_REMAINING_TEST_QUERIES = False

# The user's actual Prompt V2.1 development-set results.
V21_BASELINE = {
    "intent_accuracy": 0.68,
    "core_product_accuracy": 0.64,
    "brand_accuracy": 0.80,
    "category_accuracy": 0.42,
    "attribute_precision": 0.254902,
    "attribute_recall": 0.325000,
    "attribute_f1": 0.285714,
    "original_ndcg10": 0.799110,
    "original_mrr": 0.760000,
    "original_recall10": 0.566667,
    "original_recall50": 0.663333,
    "v21_ndcg10": 0.808263,
    "v21_mrr": 0.770000,
    "v21_recall10": 0.560000,
    "v21_recall50": 0.636667
}

## 1. Colab-safe Ollama setup

In [ ]:
import os
import time
import shutil
import subprocess
import requests
import platform
from pathlib import Path

def ollama_alive():
    try:
        return requests.get(
            f"{OLLAMA_BASE_URL}/api/tags",
            timeout=2
        ).ok
    except Exception:
        return False

def run_cmd(cmd):
    print("+", cmd)
    r = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

if shutil.which("ollama") is None:
    machine = platform.machine().lower()

    if machine in ("x86_64", "amd64"):
        arch = "amd64"
    elif machine in ("aarch64", "arm64"):
        arch = "arm64"
    else:
        raise RuntimeError(f"Unsupported architecture: {machine}")

    run_cmd("apt-get -qq update")
    run_cmd("apt-get -qq install -y zstd curl")

    url = f"https://ollama.com/download/ollama-linux-{arch}.tar.zst"

    run_cmd(
        f"curl -fL --retry 5 --retry-delay 2 "
        f"{url} -o /tmp/ollama.tar.zst"
    )

    run_cmd(
        "tar --use-compress-program=unzstd "
        "-xf /tmp/ollama.tar.zst -C /usr"
    )

if not ollama_alive():
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"

    log = open("/tmp/ollama_v22.log", "w")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
        env=env
    )

    for _ in range(90):
        if ollama_alive():
            break
        time.sleep(1)

assert ollama_alive(), "Ollama server did not start."

print("Pulling/checking:", MODEL)
subprocess.run(["ollama", "pull", MODEL], check=True)

print("Ollama ready.")

## 2. Upload inputs

In [ ]:
from pathlib import Path
import zipfile

EXPECTED = [
    "amazon_week1_v4_processed_data.zip",
    "amazon_week2_baseline_ranking_results.zip",
    "week3_manual_validation_50_normalized.csv"
]

def in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

missing = [f for f in EXPECTED if not Path(f).exists()]

if missing and in_colab():
    from google.colab import files
    print("Upload the three required files.")
    files.upload()

missing = [f for f in EXPECTED if not Path(f).exists()]

if missing:
    raise FileNotFoundError(
        "Missing: " + ", ".join(missing)
    )

WORKDIR = Path("amazon_week3_v22")
INPUT_DIR = WORKDIR / "inputs"
RESULTS_DIR = WORKDIR / "results"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for zip_name in EXPECTED[:2]:
    target = INPUT_DIR / Path(zip_name).stem
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(target)

print("Input files ready.")

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from IPython.display import display

W1_DIR = INPUT_DIR / "amazon_week1_v4_processed_data"
W2_DIR = INPUT_DIR / "amazon_week2_baseline_ranking_results"

qpr = pd.read_csv(
    W1_DIR / "query_product_relevance.csv"
)

products = pd.read_parquet(
    W1_DIR / "cleaned_products_10k.parquet"
)

fixed_test = pd.read_csv(
    W2_DIR / "fixed_test_queries.csv"
)

gold = pd.read_csv(
    "week3_manual_validation_50_normalized.csv"
)

for df in (qpr, fixed_test, gold):
    df["query_id"] = df["query_id"].astype(str)

qpr["product_id"] = qpr["product_id"].astype(str)
products["product_id"] = products["product_id"].astype(str)

query_table = (
    qpr[["query_id", "query"]]
    .drop_duplicates("query_id")
    .merge(
        fixed_test[["query_id"]],
        on="query_id",
        how="inner"
    )
)

order = {
    qid: i
    for i, qid in enumerate(
        fixed_test["query_id"].tolist()
    )
}

query_table["_order"] = (
    query_table["query_id"].map(order)
)

query_table = (
    query_table
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

dev_ids = gold["query_id"].tolist()

dev = query_table[
    query_table["query_id"].isin(dev_ids)
].copy()

dev["_order"] = dev["query_id"].map(
    {
        qid: i
        for i, qid in enumerate(dev_ids)
    }
)

dev = (
    dev
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("Development queries:", len(dev))
print("Remaining fixed-test queries:", len(query_table) - len(dev))

# 3. Prompt V2.2

The most important rule is:

> **Do not try to fill attributes. Extract only attributes that are explicitly supported by the query.**

Most queries should have only **0–4 attributes**.  
Zero attributes is a valid and often correct result.

In [ ]:
ATTRIBUTE_NAMES = [
    "color",
    "size",
    "material",
    "gender",
    "quantity",
    "style",
    "shade",
    "model",
    "product_line",
    "compatibility",
    "finish",
    "scent",
    "feature",
    "benefit",
    "spf",
    "gauge",
    "location",
    "theme",
    "skin_type",
    "volume",
    "taste",
    "hold",
    "shape",
    "ingredient",
    "target_audience",
    "purpose",
    "application"
]

QUERY_SCHEMA = {
    "type": "object",
    "properties": {
        "query": {"type": "string"},
        "intent": {
            "type": "string",
            "enum": [
                "exact_product_search",
                "product_search",
                "category_browse",
                "brand_search",
                "use_case_search",
                "compatibility_search",
                "other"
            ]
        },
        "core_product": {
            "type": ["string", "null"]
        },
        "brand": {
            "type": ["string", "null"]
        },
        "category": {
            "type": ["string", "null"]
        },
        "attributes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "enum": ATTRIBUTE_NAMES
                    },
                    "value": {
                        "type": "string"
                    },
                    "explicit": {
                        "type": "boolean"
                    }
                },
                "required": [
                    "name",
                    "value",
                    "explicit"
                ],
                "additionalProperties": False
            }
        },
        "price_min": {
            "type": ["number", "null"]
        },
        "price_max": {
            "type": ["number", "null"]
        },
        "use_case": {
            "type": ["string", "null"]
        },
        "implicit_requirements": {
            "type": "array",
            "items": {"type": "string"}
        },
        "rewritten_query": {
            "type": "string"
        }
    },
    "required": [
        "query",
        "intent",
        "core_product",
        "brand",
        "category",
        "attributes",
        "price_min",
        "price_max",
        "use_case",
        "implicit_requirements",
        "rewritten_query"
    ],
    "additionalProperties": False
}

PROMPT_V22 = """
You are a conservative e-commerce search query-understanding system.

Your highest priority is PRECISION. It is better to omit an uncertain attribute
than to output a wrong or speculative attribute.

Return exactly one JSON object following the supplied schema.

INTENT
- exact_product_search: unique named product, product line, model, shade/code,
  SKU-like identifier, or highly specific variant.
- brand_search: brand + broad category/product type without a unique item.
- use_case_search: goal/problem/outcome with no concrete product named.
- category_browse: broad category exploration.
- product_search: generic product type with explicit attributes.
- compatibility_search: compatibility is central.
- other: review fragment, subjective statement, ambiguous/incomplete text,
  or no product/category that can be identified confidently.

ATTRIBUTES: PRECISION-FIRST
- Attributes must be EXPLICITLY stated in the query.
- Every returned attribute must have explicit=true.
- Do not infer attributes from general product knowledge.
- Put inferred needs only in implicit_requirements.
- Zero attributes is valid.
- Most queries should have 0 to 4 attributes.
- Do not create an attribute merely because a token could fit an attribute name.

NO DUPLICATION
Never place these top-level concepts inside attributes:
- brand
- category
- core product / product type
- use case
- price

ATOMIC ATTRIBUTE RULE
One attribute object must contain exactly ONE atomic value/concept.

WRONG:
{"name":"ingredient","value":"mint, myrrh"}
{"name":"feature","value":"organic, wildcrafted"}

RIGHT:
{"name":"ingredient","value":"mint"}
{"name":"ingredient","value":"myrrh"}
{"name":"feature","value":"organic"}
{"name":"feature","value":"wildcrafted"}

If a query explicitly contains several values for the same attribute,
return several attribute objects.

PRODUCT IDENTITY VS ATTRIBUTE
- A product name is not a model.
- A brand is not a product_line.
- A category is not a feature.
- A product name must never be copied into model/product_line merely to fill a field.
- model is only for an explicit model/SKU/code such as "1090", "PMS061", "Surf-350".
- product_line is only for an explicitly named family/line such as "Nice 'n Easy".
- shade should combine an explicit shade code and shade name when they belong together,
  e.g. "765 medium".

NUMBERS / IDENTIFIERS
Preserve exact numeric information.
- 33.8 oz -> size "33.8oz"
- 18.75 oz -> size "18.75oz"
- 20 volume -> volume "20 volume"
- 5.3 may be a shade code; do not treat it as quantity unless the query says quantity.
- 765 medium -> one shade value "765 medium"
- PMS061 -> model "PMS061"
- Surf-350 -> model "Surf-350"

Do not split decimals.
Do not reinterpret identifiers as size/quantity/price without evidence.

REVIEW FRAGMENTS / AMBIGUOUS TEXT
For an ambiguous fragment such as:
"set don't help"
return intent="other", core_product=null, category=null, attributes=[].

For a review-like phrase such as:
"great hold"
you may extract hold="great" ONLY because "hold" is explicitly named.
Do not infer hair product, styling product, gel, spray, or category.

REWRITE
- Preserve every explicit brand, product line, model, shade/code, decimal,
  size/unit, quantity, and compatibility term.
- Remove filler only.
- Do not add speculative product/category terms.
- Do not change product type.
- For intent="other" with no identifiable product, keep the rewrite very close
  to the original instead of inventing a search query.

Output JSON only.
""".strip()

print(PROMPT_V22)

In [ ]:
import requests
import time
from tqdm.auto import tqdm

def call_v22(query, retries=3):
    messages = [
        {
            "role": "system",
            "content": PROMPT_V22
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]

    last_error = None

    for attempt in range(1, retries + 1):
        start = time.perf_counter()

        payload = {
            "model": MODEL,
            "messages": messages,
            "stream": False,
            "format": QUERY_SCHEMA,
            "options": {
                "temperature": 0,
                "seed": RANDOM_SEED,
                "num_ctx": 4096
            }
        }

        try:
            r = requests.post(
                f"{OLLAMA_BASE_URL}/api/chat",
                json=payload,
                timeout=180
            )

            if r.status_code >= 400:
                payload["format"] = "json"
                r = requests.post(
                    f"{OLLAMA_BASE_URL}/api/chat",
                    json=payload,
                    timeout=180
                )

            r.raise_for_status()

            obj = json.loads(
                r.json()["message"]["content"]
            )
            obj["query"] = query

            # Extra programmatic precision guard.
            attrs = obj.get("attributes", [])
            clean_attrs = []

            if isinstance(attrs, list):
                for a in attrs:
                    if not isinstance(a, dict):
                        continue

                    name = str(a.get("name", "")).strip()
                    value = str(a.get("value", "")).strip()

                    if (
                        name in ATTRIBUTE_NAMES
                        and value
                        and a.get("explicit") is True
                    ):
                        clean_attrs.append({
                            "name": name,
                            "value": value,
                            "explicit": True
                        })

            obj["attributes"] = clean_attrs

            return (
                obj,
                time.perf_counter() - start,
                None
            )

        except Exception as e:
            last_error = repr(e)
            time.sleep(min(2 * attempt, 6))

    return None, np.nan, last_error

## 4. Run V2.2 on the same 50-query development set

In [ ]:
rows = []

for r in tqdm(
    dev.itertuples(index=False),
    total=len(dev),
    desc="Prompt V2.2 development run"
):
    obj, latency, error = call_v22(r.query)
    obj = obj or {}

    rows.append({
        "query_id": r.query_id,
        "query": r.query,
        "v22_intent": obj.get("intent"),
        "v22_core_product":
            obj.get("core_product"),
        "v22_brand":
            obj.get("brand"),
        "v22_category":
            obj.get("category"),
        "v22_attributes_json":
            json.dumps(
                obj.get("attributes", []),
                ensure_ascii=False
            ),
        "v22_price_min":
            obj.get("price_min"),
        "v22_price_max":
            obj.get("price_max"),
        "v22_use_case":
            obj.get("use_case"),
        "v22_implicit_requirements_json":
            json.dumps(
                obj.get(
                    "implicit_requirements",
                    []
                ),
                ensure_ascii=False
            ),
        "v22_rewritten_query":
            obj.get("rewritten_query"),
        "v22_latency_sec":
            latency,
        "v22_error":
            error
    })

v22 = pd.DataFrame(rows)

v22.to_csv(
    RESULTS_DIR / "week3_prompt_v22_outputs_50.csv",
    index=False
)

display(v22.head(10))

## 5. Strict normalized manual evaluation

This uses the same canonical gold labels as V2.1.  
The primary Attribute F1 remains a strict pair-level metric.

In [ ]:
def norm_text(x):
    if pd.isna(x):
        return ""

    s = (
        str(x)
        .strip()
        .lower()
        .replace("’", "'")
        .replace("–", "-")
    )

    s = re.sub(
        r"\s+",
        " ",
        s
    )

    s = re.sub(
        r"(\d)\s+(oz|g|ml|lb|lbs)\b",
        r"\1\2",
        s
    )

    return s

def load_attr_list(x):
    try:
        out = json.loads(x) if isinstance(x, str) and x.strip() else []
        return out if isinstance(out, list) else []
    except Exception:
        return []

def attr_pairs(x):
    pairs = set()

    for item in load_attr_list(x):
        if not isinstance(item, dict):
            continue

        name = norm_text(
            item.get("name")
        )

        value = norm_text(
            item.get("value")
        )

        if name and value:
            pairs.add((name, value))

    return pairs

merged = gold.merge(
    v22,
    on=["query_id", "query"],
    how="left"
)

def exact_acc(pred_col, true_col):
    return np.mean([
        norm_text(p) == norm_text(t)
        for p, t in zip(
            merged[pred_col],
            merged[true_col]
        )
    ])

tp = fp = fn = 0

for _, r in merged.iterrows():
    pred = attr_pairs(
        r["v22_attributes_json"]
    )

    truth = attr_pairs(
        r["true_attributes_normalized_json"]
    )

    tp += len(pred & truth)
    fp += len(pred - truth)
    fn += len(truth - pred)

precision = (
    tp / (tp + fp)
    if tp + fp
    else 0.0
)

recall = (
    tp / (tp + fn)
    if tp + fn
    else 0.0
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if precision + recall
    else 0.0
)

metrics = pd.DataFrame({
    "metric": [
        "intent_accuracy",
        "core_product_accuracy",
        "brand_accuracy",
        "category_accuracy",
        "attribute_precision_normalized",
        "attribute_recall_normalized",
        "attribute_f1_normalized"
    ],
    "v21": [
        V21_BASELINE["intent_accuracy"],
        V21_BASELINE["core_product_accuracy"],
        V21_BASELINE["brand_accuracy"],
        V21_BASELINE["category_accuracy"],
        V21_BASELINE["attribute_precision"],
        V21_BASELINE["attribute_recall"],
        V21_BASELINE["attribute_f1"]
    ],
    "v22": [
        exact_acc(
            "v22_intent",
            "true_intent"
        ),
        exact_acc(
            "v22_core_product",
            "true_core_product"
        ),
        exact_acc(
            "v22_brand",
            "true_brand"
        ),
        exact_acc(
            "v22_category",
            "true_category"
        ),
        precision,
        recall,
        f1
    ]
})

metrics["delta_v22_minus_v21"] = (
    metrics["v22"]
    - metrics["v21"]
)

metrics.to_csv(
    RESULTS_DIR
    / "week3_prompt_v22_manual_metrics.csv",
    index=False
)

display(metrics)

## 6. Attribute-count diagnostics

This directly checks whether over-extraction was reduced.

In [ ]:
def count_attrs(x):
    return len(load_attr_list(x))

merged["gold_attribute_count"] = (
    merged[
        "true_attributes_normalized_json"
    ].apply(count_attrs)
)

merged["v22_attribute_count"] = (
    merged[
        "v22_attributes_json"
    ].apply(count_attrs)
)

count_summary = pd.DataFrame({
    "metric": [
        "mean_gold_attributes_per_query",
        "mean_v22_attributes_per_query",
        "queries_gold_zero_attributes",
        "queries_v22_zero_attributes",
        "total_gold_attribute_pairs",
        "total_v22_attribute_pairs"
    ],
    "value": [
        merged["gold_attribute_count"].mean(),
        merged["v22_attribute_count"].mean(),
        (merged["gold_attribute_count"] == 0).sum(),
        (merged["v22_attribute_count"] == 0).sum(),
        merged["gold_attribute_count"].sum(),
        merged["v22_attribute_count"].sum()
    ]
})

count_summary.to_csv(
    RESULTS_DIR
    / "week3_prompt_v22_attribute_count_diagnostics.csv",
    index=False
)

display(count_summary)

## 7. Error analysis

In [ ]:
def diff_attributes(row):
    truth = attr_pairs(
        row[
            "true_attributes_normalized_json"
        ]
    )

    pred = attr_pairs(
        row["v22_attributes_json"]
    )

    return pd.Series({
        "missing_attributes":
            json.dumps(
                sorted(truth - pred),
                ensure_ascii=False
            ),
        "extra_attributes":
            json.dumps(
                sorted(pred - truth),
                ensure_ascii=False
            )
    })

diff = merged.apply(
    diff_attributes,
    axis=1
)

error_table = pd.concat(
    [
        merged[
            [
                "query_id",
                "query",
                "true_intent",
                "v22_intent",
                "true_core_product",
                "v22_core_product",
                "true_attributes_normalized_json",
                "v22_attributes_json",
                "v22_rewritten_query",
                "notes"
            ]
        ],
        diff
    ],
    axis=1
)

error_table.to_csv(
    RESULTS_DIR
    / "week3_prompt_v22_error_analysis.csv",
    index=False
)

print("Remaining attribute errors:")
display(
    error_table[
        (
            error_table[
                "missing_attributes"
            ] != "[]"
        )
        |
        (
            error_table[
                "extra_attributes"
            ] != "[]"
        )
    ].head(30)
)

# 8. BM25 retrieval — Original vs V2.2 Rewrite

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score

def tokenize(text):
    if pd.isna(text):
        return []

    return re.findall(
        r"[a-z0-9]+",
        str(text).lower()
    )

products["title"] = (
    products["title"]
    .fillna("")
    .astype(str)
)

products["description"] = (
    products["description"]
    .fillna("")
    .astype(str)
)

documents = (
    products["title"].str.strip()
    + " "
    + products["description"].str.strip()
).str.strip()

bm25 = BM25Okapi(
    [
        tokenize(x)
        for x in tqdm(
            documents,
            desc="Tokenizing catalog"
        )
    ]
)

product_ids = (
    products["product_id"]
    .tolist()
)

pid_to_idx = {
    pid: i
    for i, pid in enumerate(
        product_ids
    )
}

def reciprocal_rank(labels):
    for i, y in enumerate(
        labels,
        start=1
    ):
        if y >= 1:
            return 1.0 / i

    return 0.0

def recall_at_k(
    retrieved,
    relevant,
    k
):
    relevant = set(relevant)

    if not relevant:
        return np.nan

    return (
        len(
            set(retrieved[:k])
            & relevant
        )
        / len(relevant)
    )

def score_bm25(q):
    return np.asarray(
        bm25.get_scores(
            tokenize(q)
        )
    )

def evaluate_version(
    query_id,
    query_text,
    version
):
    scores = score_bm25(
        query_text
    )

    judged = qpr[
        qpr["query_id"]
        == str(query_id)
    ].copy()

    judged["idx"] = (
        judged["product_id"]
        .map(pid_to_idx)
    )

    judged = judged[
        judged["idx"].notna()
    ].copy()

    judged["idx"] = (
        judged["idx"]
        .astype(int)
    )

    judged["score"] = scores[
        judged["idx"].values
    ]

    ranked = judged.sort_values(
        "score",
        ascending=False
    )

    labels = (
        ranked["relevance_label"]
        .astype(float)
        .tolist()
    )

    ndcg10 = float(
        ndcg_score(
            judged[
                "relevance_label"
            ]
            .astype(float)
            .to_numpy()
            .reshape(1, -1),
            judged[
                "score"
            ]
            .astype(float)
            .to_numpy()
            .reshape(1, -1),
            k=10
        )
    )

    top_idx = np.argpartition(
        scores,
        -50
    )[-50:]

    top_idx = top_idx[
        np.argsort(
            scores[top_idx]
        )[::-1]
    ]

    retrieved = [
        product_ids[i]
        for i in top_idx
    ]

    strong = set(
        judged.loc[
            judged[
                "relevance_label"
            ] >= 2,
            "product_id"
        ].astype(str)
    )

    return {
        "query_id": str(query_id),
        "version": version,
        "ndcg@10": ndcg10,
        "mrr": reciprocal_rank(labels),
        "recall@10": recall_at_k(
            retrieved,
            strong,
            10
        ),
        "recall@50": recall_at_k(
            retrieved,
            strong,
            50
        )
    }

retrieval_rows = []

for r in tqdm(
    merged.itertuples(index=False),
    total=len(merged),
    desc="BM25 V2.2 evaluation"
):
    retrieval_rows.append(
        evaluate_version(
            r.query_id,
            r.query,
            "original"
        )
    )

    retrieval_rows.append(
        evaluate_version(
            r.query_id,
            r.v22_rewritten_query,
            "v22_rewrite"
        )
    )

retrieval = pd.DataFrame(
    retrieval_rows
)

retrieval_summary = (
    retrieval
    .groupby("version")[
        [
            "ndcg@10",
            "mrr",
            "recall@10",
            "recall@50"
        ]
    ]
    .mean()
    .reset_index()
)

retrieval.to_csv(
    RESULTS_DIR
    / "week3_prompt_v22_retrieval_per_query.csv",
    index=False
)

retrieval_summary.to_csv(
    RESULTS_DIR
    / "week3_prompt_v22_retrieval_summary.csv",
    index=False
)

display(retrieval_summary)

## 9. Final V2.2 development gate

In [ ]:
intent_accuracy = float(
    metrics.loc[
        metrics["metric"]
        == "intent_accuracy",
        "v22"
    ].iloc[0]
)

attribute_f1 = float(
    metrics.loc[
        metrics["metric"]
        == "attribute_f1_normalized",
        "v22"
    ].iloc[0]
)

original_r50 = float(
    retrieval_summary.loc[
        retrieval_summary["version"]
        == "original",
        "recall@50"
    ].iloc[0]
)

v22_r50 = float(
    retrieval_summary.loc[
        retrieval_summary["version"]
        == "v22_rewrite",
        "recall@50"
    ].iloc[0]
)

recall_floor = (
    original_r50
    * (1 - MAX_RECALL50_RELATIVE_DROP)
)

gate = (
    attribute_f1
    >= TARGET_ATTRIBUTE_F1
    and intent_accuracy
    >= TARGET_INTENT_ACCURACY
    and v22_r50
    >= recall_floor
)

print(
    "Intent Accuracy:",
    round(intent_accuracy, 4)
)

print(
    "Normalized Attribute F1:",
    round(attribute_f1, 4)
)

print(
    "Original Recall@50:",
    round(original_r50, 4)
)

print(
    "V2.2 Rewrite Recall@50:",
    round(v22_r50, 4)
)

print(
    "Recall@50 required floor:",
    round(recall_floor, 4)
)

print()

print(
    "DEVELOPMENT GATE:",
    (
        "PASS — ready for remaining fixed-test queries"
        if gate
        else
        "NOT PASSED — report the limitation rather than endlessly tuning"
    )
)

# 10. Optional remaining fixed-test run

Only enable this if V2.2 passes the development gate.

In [ ]:
if RUN_REMAINING_TEST_QUERIES and gate:
    holdout = query_table[
        ~query_table[
            "query_id"
        ].isin(dev_ids)
    ].copy()

    holdout_rows = []

    for r in tqdm(
        holdout.itertuples(index=False),
        total=len(holdout),
        desc="V2.2 remaining fixed-test queries"
    ):
        obj, latency, error = call_v22(
            r.query
        )

        obj = obj or {}

        holdout_rows.append({
            "query_id":
                r.query_id,
            "query":
                r.query,
            "intent":
                obj.get("intent"),
            "core_product":
                obj.get("core_product"),
            "brand":
                obj.get("brand"),
            "category":
                obj.get("category"),
            "attributes_json":
                json.dumps(
                    obj.get(
                        "attributes",
                        []
                    ),
                    ensure_ascii=False
                ),
            "rewritten_query":
                obj.get(
                    "rewritten_query"
                ),
            "latency_sec":
                latency,
            "error":
                error
        })

    holdout_df = pd.DataFrame(
        holdout_rows
    )

    holdout_df.to_csv(
        RESULTS_DIR
        / "week3_prompt_v22_remaining_test_queries.csv",
        index=False
    )

    holdout_df.to_parquet(
        RESULTS_DIR
        / "week3_prompt_v22_remaining_test_queries.parquet",
        index=False
    )

    print(
        "Remaining test queries processed:",
        len(holdout_df)
    )

elif RUN_REMAINING_TEST_QUERIES and not gate:
    print(
        "Gate did not pass. "
        "Holdout run intentionally blocked."
    )

else:
    print(
        "Holdout run disabled."
    )

## 11. Export V2.2 results

In [ ]:
from datetime import datetime
import shutil

v22_ndcg = float(
    retrieval_summary.loc[
        retrieval_summary["version"]
        == "v22_rewrite",
        "ndcg@10"
    ].iloc[0]
)

v22_mrr = float(
    retrieval_summary.loc[
        retrieval_summary["version"]
        == "v22_rewrite",
        "mrr"
    ].iloc[0]
)

report = f"""
# Amazon Search / LTR Project
## Week 3 Prompt V2.2 — Precision-First Atomic Attributes

### V2.1 Baseline
- Intent Accuracy: {V21_BASELINE['intent_accuracy']:.4f}
- Attribute Precision: {V21_BASELINE['attribute_precision']:.4f}
- Attribute Recall: {V21_BASELINE['attribute_recall']:.4f}
- Attribute F1: {V21_BASELINE['attribute_f1']:.4f}
- Rewrite Recall@50: {V21_BASELINE['v21_recall50']:.4f}

### V2.2 Development Results
- Intent Accuracy: {intent_accuracy:.4f}
- Attribute Precision: {precision:.4f}
- Attribute Recall: {recall:.4f}
- Attribute F1: {attribute_f1:.4f}
- Rewrite NDCG@10: {v22_ndcg:.4f}
- Rewrite MRR: {v22_mrr:.4f}
- Original Recall@50: {original_r50:.4f}
- V2.2 Rewrite Recall@50: {v22_r50:.4f}

### Gate
{'PASS' if gate else 'NOT PASSED'}

### Interpretation
V2.2 intentionally optimizes attribute precision by requiring explicit, atomic
attribute values and preventing duplication of product identity fields. If the
0.75 F1 target is still not reached, the result should be reported as a model/
data limitation instead of repeatedly tuning on the same 50-query development set.

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
""".strip()

(
    RESULTS_DIR
    / "Amazon_Week3_PromptV22_Report.md"
).write_text(
    report + "\n",
    encoding="utf-8"
)

(
    RESULTS_DIR
    / "prompt_v22.txt"
).write_text(
    PROMPT_V22 + "\n",
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    "amazon_week3_prompt_v22_results",
    "zip",
    root_dir=RESULTS_DIR
)

print(report)
print()
print("Created:", zip_path)

if in_colab():
    from google.colab import files
    files.download(zip_path)